# MOE8-DSS Kaggle execution

This notebook runs a pinned DSS source commit. The first submission is a two-epoch GPU smoke run. After checking its logs and artifacts, change RUN_MODE to "full" for one fresh 30-epoch experiment.

In [ ]:
BRANCH = "experiment/moe8-domain-wise-spearman"
SOURCE_COMMIT = "00ec238d6e280cfd319fc3317f3bc02bd2942732"
DATASET_NAME = "XACLE_2027_data"
ARTIFACTS_NAME = "moescore-full8-artifacts"
DATASET_ROOT_OVERRIDE = None
ARTIFACTS_ROOT_OVERRIDE = None
RESUME_ARTIFACTS_OVERRIDE = None
RUN_MODE = "smoke"  # smoke or full; initial push must satisfy smoke gate
DEVICE = "cuda:0"
SEED = 2027
SMOKE_EPOCHS = 2
MAX_EPOCHS = 30
OUTPUT_DIR = "/kaggle/working/moescore_full8_dss_artifacts"

assert RUN_MODE in ("smoke", "full") and SEED == 2027 and SMOKE_EPOCHS == 2 and MAX_EPOCHS == 30
assert DATASET_NAME == "XACLE_2027_data" and ARTIFACTS_NAME == "moescore-full8-artifacts"


In [ ]:
import json, os, platform, shutil, subprocess, sys
from pathlib import Path
import numpy as np
import scipy
import torch
print({"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "scipy": scipy.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), "DSS Kaggle run requires an available GPU"


In [ ]:
REPO_URL = "https://github.com/vietquang8071/MOESCORE.git"
SOURCE_ROOT = Path("/kaggle/working/moescore_dss_source")
if SOURCE_ROOT.exists():
    raise FileExistsError(f"Source checkout already exists: {SOURCE_ROOT}")
subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(SOURCE_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=SOURCE_ROOT, check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=SOURCE_ROOT, text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
sys.path.insert(0, str(SOURCE_ROOT))
print({"source_commit": actual_commit})


In [ ]:
from scipy.optimize import isotonic_regression
from loss.l2_soft_rank import l2_soft_rank
probe = torch.tensor([0., .02, .04, .06], dtype=torch.float64, device=DEVICE, requires_grad=True)
ranks = l2_soft_rank(probe, 0.1)
expected = 2.5 + (probe - probe.mean()) / .1
assert torch.allclose(ranks, expected, atol=1e-9)
ranks[0].backward()
assert torch.isfinite(probe.grad).all()
print({"ranking_backend": "internal_exact_pav_l2_v1", "rank_probe": ranks.detach().cpu().tolist()})


In [ ]:
from scripts.diagnostics.d01_moe8 import resolve_dataset_mount, discover_frozen_artifacts
INPUT_ROOT = Path("/kaggle/input")
DATASET_ROOT = Path(DATASET_ROOT_OVERRIDE) if DATASET_ROOT_OVERRIDE else resolve_dataset_mount(INPUT_ROOT, "xacle-2027-data")
ARTIFACTS_ROOT = Path(ARTIFACTS_ROOT_OVERRIDE) if ARTIFACTS_ROOT_OVERRIDE else resolve_dataset_mount(INPUT_ROOT, "moescore-full8-artifacts")
assert DATASET_ROOT.is_dir() and ARTIFACTS_ROOT.is_dir()
artifacts = discover_frozen_artifacts(ARTIFACTS_ROOT)
print({"dataset_root": str(DATASET_ROOT), "artifacts_root": str(ARTIFACTS_ROOT),
       "feature_store": str(artifacts["feature_store"]), "dataset_ids": ["qnv8071/xacle-2027-data", "qnv8071/moescore-full8-artifacts"],
       "dataset_versions": "not exposed by mounted files"})


In [ ]:
# The CLI preflight validates all shards, official metadata, every WAV hash, and provenance before creating the gate.
# It writes the evidence to preflight_report.json; a failure stops the job before optimization.
print("DSS cache and official-data preflight is the first stage of the training CLI.")


In [ ]:
RUN_OUTPUT = Path(OUTPUT_DIR if RUN_MODE == "full" else OUTPUT_DIR.replace("_artifacts", "_smoke_artifacts"))
resume_path = None
if RESUME_ARTIFACTS_OVERRIDE:
    resume_root = Path(RESUME_ARTIFACTS_OVERRIDE)
    source_checkpoints = resume_root / "checkpoints"
    (RUN_OUTPUT / "checkpoints").mkdir(parents=True, exist_ok=True)
    for name in ("best.pt", "last.pt"):
        shutil.copy2(source_checkpoints / name, RUN_OUTPUT / "checkpoints" / name)
    resume_path = RUN_OUTPUT / "checkpoints" / "last.pt"
command = [sys.executable, str(SOURCE_ROOT / "scripts/train_moe8_dss.py"),
           "--dataset-root", str(DATASET_ROOT), "--artifacts-root", str(ARTIFACTS_ROOT),
           "--output-dir", str(RUN_OUTPUT), "--run-mode", RUN_MODE, "--device", DEVICE]
if resume_path:
    command.extend(["--resume", str(resume_path)])
print({"run_mode": RUN_MODE, "output_dir": str(RUN_OUTPUT), "source_commit": actual_commit})
subprocess.run(command, cwd=SOURCE_ROOT, check=True)


In [ ]:
import csv
metadata = json.loads((RUN_OUTPUT / "experiment_metadata.json").read_text())
assert (RUN_OUTPUT / "preflight_report.json").is_file()
assert (RUN_OUTPUT / "checkpoints/best.pt").is_file() and (RUN_OUTPUT / "checkpoints/last.pt").is_file()
with (RUN_OUTPUT / "training_history.csv").open() as stream:
    history = list(csv.DictReader(stream))
expected_epochs = SMOKE_EPOCHS if RUN_MODE == "smoke" else MAX_EPOCHS
assert len(history) == expected_epochs and metadata["optimizer_steps"] == 59 * expected_epochs
assert all(np.isfinite(float(row["total_loss"])) and np.isfinite(float(row["gradient_norm"])) for row in history)
assert all(float(row["epoch_gate_update_l2"]) > 0 for row in history)
assert np.isfinite(float(metadata["best_validation_srcc"]))
if RUN_MODE == "smoke":
    assert metadata["smoke_status"] == "COMPLETE" and metadata["status"] == "PARTIAL"
    assert not (RUN_OUTPUT / "predictions/public_test.csv").exists()
else:
    assert metadata["status"] == "COMPLETE" and metadata["prediction_count"] == 13500
    for split, count in (("train", 7500), ("validation", 3000), ("public_test", 3000)):
        with (RUN_OUTPUT / "predictions" / f"{split}.csv").open() as stream:
            assert sum(1 for _ in csv.DictReader(stream)) == count
print({"status": metadata["status"], "smoke_status": metadata.get("smoke_status"),
       "epochs": len(history), "optimizer_steps": metadata["optimizer_steps"],
       "best_epoch": metadata["best_epoch"], "best_validation_srcc": metadata["best_validation_srcc"],
       "best_checkpoint_sha256": metadata["best_checkpoint_sha256"],
       "peak_gpu_memory_bytes": metadata["peak_gpu_memory_bytes"], "output_dir": str(RUN_OUTPUT)})
